In [1]:

import sys

from pathlib import Path

print(sys.executable)

print(Path.cwd())

c:\dev\llm-data-analysis-course\.venv-new\Scripts\python.exe
c:\dev\llm-data-analysis-course\notebooks\ch09


## STEP 1. 예측 문제와 예측 시점 정의

무엇을 예측하는가?

예측 단위는 무엇인가?

언제 예측하는가?

그 시점에 실제로 알 수 있는 정보는 무엇인가?

- **Target (예측 대상)**: `order_total` (총 주문 금액)
- **예측 단위**: 주문 1건
- **예측 시점**: 주문 시점 (주문 생성 순간)
- **예측 시점에 알 수 있는 정보**: 주문 메타데이터(주문 월/요일 등), 고객 비식별 정보(나이, 성별, 도시 등), 결제 수단
- **예측 시점에 알 수 없는 정보**: 주문 상세 수량(`quantity`), 단가(`unit_price`), 품목별 금액(`line_total`), 주문 상태(`order_status`)
- **예측 목적**: 주문 생성 시점에 고객 및 주문 특성만으로 예상 주문 금액을 예측하여 고객군 분석 및 프로모션 기획에 활용

## STEP 2. Feature Leakage Audit (데이터 누수 점검)

### 1. 전체 Feature 사용 가능 여부 점검표

| No | Feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보 / ID | 최종 사용 | 판단 이유 |
| :-: | :--- | :---: | :---: | :---: | :---: | :--- |
| 1 | `order_month` | O | X | X | **사용** | 주문 생성 시점에 알 수 있는 월 정보 |
| 2 | `order_dayofweek` | O | X | X | **사용** | 주문 생성 시점에 알 수 있는 요일 정보 |
| 3 | `age` | O | X | X | **사용** | 고객 비식별 연령대 프로필 |
| 4 | `gender` | O | X | X | **사용** | 고객 비식별 성별 프로필 |
| 5 | `city` | O | X | X | **사용** | 고객 거주/배송 도시 정보 |
| 6 | `payment_method` | O | X | X | **사용** | 결제 수단 선택 정보 |
| 7 | `order_total` | - | O | - | **제외** | Target (예측 대상 그 자체) |
| 8 | `line_total` | - | O | - | **제외** | Target 계산에 사용되는 품목별 합계 |
| 9 | `quantity` | X | O | - | **제외** | Target 계산 직접 재료 (주문 수량) |
| 10 | `unit_price` | X | O | - | **제외** | Target 계산 직접 재료 (개당 단가) |
| 11 | `item_count` | X | O | - | **제외** | Target 관련 파생 변수 (주문 품목 수) |
| 12 | `total_quantity` | X | O | - | **제외** | Target 관련 파생 변수 (총 구매 수량) |
| 13 | `avg_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (평균 단가) |
| 14 | `min_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (최저 단가) |
| 15 | `max_unit_price` | X | O | - | **제외** | Target 관련 파생 변수 (최고 단가) |
| 16 | `order_status` | X | X | O | **제외** | 주문 완료/취소/배송 등 사후 상태 정보 |
| 17 | `order_id` | - | X | O | **제외** | 주문 식별자 (ID) |
| 18 | `customer_id` | - | X | O | **제외** | 고객 식별자 (ID) |
| 19 | `product_id` | - | X | O | **제외** | 상품 식별자 (ID) |
| 20 | `created_at` | X | X | O | **제외** | 주문 생성 정확한 타임스탬프 (메타데이터) |
| 21 | `updated_at` | X | X | O | **제외** | 주문 상태 변경 타임스탬프 (사후 정보) |
| 22 | `order_date` | - | X | O | **제외** | 시간 분할 기준 컬럼 (Feature 입력 불요) |
| 23 | `category_id` | X | X | O | **제외** | 상품 카테고리 식별자 |

---

### 2. 점검 요약 및 원칙
- **허용 Feature**: `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city` (총 6개)
- **금지 Feature**: 
  - **Target 관련**: `order_total`, `line_total`, `quantity`, `unit_price`, `item_count`, `total_quantity`, `avg_unit_price`
  - **사후 정보**: `order_status`
  - **식별자**: `order_id`, `customer_id`, `product_id`
- **원칙**: 실제 서비스에서 **주문 생성 순간**에 알 수 없는 정보나 정답의 재료, 단순 ID는 데이터셋에 존재하더라도 모델 입력(Feature)에서 제외합니다.

이번 단계는 **'데이터 누수(Data Leakage) 점검'** 과정입니다.

시험을 볼 때 문제지에 정답이 슬쩍 적혀 있으면 실력을 정확히 측정할 수 없는 것처럼, 머신러닝도 **미래의 정보나 정답의 재료**가 미리 들어간 채 학습하면 '가짜 성능'이 나오게 됩니다.

따라서 실제 **주문이 생성되는 순간**에 알 수 있는 정보만 모델에게 주고, 결제 후 상태나 정답 계산에 쓰인 정보는 미리 제외하는 작업을 거치는 것입니다. 그래야 실전에 투입했을 때도 오차 없이 진짜 예측을 해낼 수 있습니다.

## STEP 2. Feature Leakage Audit (데이터 누수 점검) 결과 요약

### 1. 점검 결과 요약표

| 구분 | 포함된 Feature 목록 | 개수 | 최종 처리 |
| :--- | :--- | :---: | :---: |
| **허용 Feature (Inputs)** | `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city` | 6개 | **모델 입력(학습) 사용** |
| **Target (예측 대상)** | `order_total` | 1개 | **Target으로 설정** |
| **Target 계산 재료** | `line_total`, `quantity`, `unit_price`, `item_count`, `total_quantity`, `avg_unit_price` | 6개 | **모델 입력에서 제외** |
| **사후 정보** | `order_status` | 1개 | **모델 입력에서 제외** |
| **단순 식별자 (ID)** | `order_id`, `customer_id`, `product_id` | 3개 | **모델 입력에서 제외** |

---

### 2. 누수 점검 판단 근거 및 원칙

1. **예측 시점의 타당성**: 주문 생성 순간 고객 프로필(`age`, `gender`, `city`)과 주문 기본 선택 정보(`order_month`, `order_dayofweek`, `payment_method`)만 알 수 있으므로 해당 6개 변수만 입력으로 허용합니다.
2. **Target 직접/간접 재료 차단**: `order_total` 계산에 직접 쓰이는 `quantity`, `unit_price` 및 이들의 파생 변수(`item_count` 등)는 정답을 미리 알려주는 역할을 하므로 완전히 제외합니다.
3. **사후 정보 차단**: `order_status`(주문 상태)는 주문 결제 및 이행 이후 생성되는 사후 정보이므로 예측 시점에 사용할 수 없습니다.
4. **식별자 차단**: `order_id`, `customer_id`, `product_id` 등은 단순 인덱스/식별자 역할을 하므로 예측 수식에서 제외합니다.